# Session 4 — Reflexion (Learning Agents)

**GOAL:** Build an agent that grades its own work and saves lessons learned to "Memory" so it never makes the same mistake twice.

The Loop:
1. **ACTOR** drafts an email.
2. **EVALUATOR** critiques it (Pass/Fail) based on a rubric.
3. If Fail → Actor tries again using the critique.
4. If Pass → Save the task + final output to MEMORY.

In [ ]:
import os, sys, json, warnings, logging

warnings.filterwarnings('ignore')
logging.getLogger().setLevel(logging.ERROR)
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from dotenv import load_dotenv
load_dotenv(os.path.join(PROJECT_ROOT, '.env'))

try:
    import utils.dns_patch
except Exception:
    pass

from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import create_react_agent
from typing import TypedDict
from utils.tools import fetch_emails, draft_email_reply
from utils.llm_router import get_routed_llm

print('Setup complete.')

### Episodic Memory (The "Brain" File)

Instead of fine-tuning the model's weights, we save successful workflows to a local JSON file.  
On future runs, the agent reads this file to remember how it succeeded in the past.

In [ ]:
MEMORY_FILE = os.path.join(os.path.abspath('.'), 'episodic_memory.json')

def load_memory():
    if os.path.exists(MEMORY_FILE):
        with open(MEMORY_FILE, 'r') as f:
            return json.load(f)
    return []

def save_memory(task, successful_result, critique):
    memory = load_memory()
    memory.append({'task': task, 'successful_result': successful_result, 'lesson_learned': critique})
    with open(MEMORY_FILE, 'w') as f:
        json.dump(memory, f, indent=2)

print(f'Memory file: {MEMORY_FILE}')
print(f'Existing memories: {len(load_memory())}')

### Graph State & Agents

In [ ]:
class ReflexionState(TypedDict):
    task: str
    current_draft: str
    critique: str
    is_passing: bool
    iterations: int

llm = get_routed_llm(role='master_model')

actor_agent = create_react_agent(
    llm,
    tools=[fetch_emails, draft_email_reply],
    prompt=SystemMessage(content=(
        'You are the DRAFTER AGENT.\n'
        'Your job is to read the task and output a professional email draft.\n'
        'If you are given a critique, you MUST rewrite the draft to address it.'
    )),
)

evaluator_agent = create_react_agent(
    llm,
    tools=[],
    prompt=SystemMessage(content=(
        'You are the EVALUATOR AGENT.\n'
        'Analyze the provided email draft against this strict rubric:\n'
        '  1. Tone: Must be exceptionally professional and polite.\n'
        '  2. Length: Must be under 100 words.\n'
        '  3. Structure: Must have a clear greeting, body, and sign-off.\n\n'
        'Respond with EXACTLY this JSON format:\n'
        '{"pass": true/false, "critique": "detailed feedback"}'
    )),
)

print('Agents created: Actor (Drafter) + Evaluator')

### Define Graph Nodes

In [ ]:
def actor_node(state: ReflexionState):
    print(f'\n[ACTOR] Drafting (Iteration {state["iterations"] + 1}) ...')
    memories = load_memory()
    memory_context = ''
    if memories:
        memory_context = 'PAST SUCCESSFUL EXAMPLES TO LEARN FROM:\n'
        for i, m in enumerate(memories[-3:]):
            memory_context += f'Example {i+1}: {m["successful_result"]}\n'
    prompt = f'Task: {state["task"]}\n\n{memory_context}'
    if state['critique']:
        prompt += f'\n\nYour last draft was REJECTED. Critique: {state["critique"]}\nRewrite it to be better.'
    result = actor_agent.invoke({'messages': [HumanMessage(content=prompt)]})
    last_msg = result['messages'][-1].content
    return {'current_draft': str(last_msg), 'iterations': state['iterations'] + 1}

def evaluator_node(state: ReflexionState):
    print('\n[EVALUATOR] Grading draft ...')
    prompt = f'Task: {state["task"]}\n\nDraft to evaluate:\n{state["current_draft"]}'
    result = evaluator_agent.invoke({'messages': [HumanMessage(content=prompt)]})
    raw_response = result['messages'][-1].content
    try:
        if '```json' in raw_response:
            json_str = raw_response.split('```json')[1].split('```')[0]
        elif '```' in raw_response:
            json_str = raw_response.split('```')[1].split('```')[0]
        else:
            json_str = raw_response
        evaluation = json.loads(json_str.strip())
        is_pass = evaluation.get('pass', False)
        critique = evaluation.get('critique', 'No critique provided.')
    except Exception as e:
        print(f'   Warning: Failed to parse evaluator JSON: {e}')
        is_pass = False
        critique = 'Failed to evaluate properly. Please try again.'
    print(f'   Result: {"PASS" if is_pass else "FAIL"}')
    print(f'   Critique: {critique}')
    return {'is_passing': is_pass, 'critique': critique}

def route_evaluation(state: ReflexionState):
    if state['is_passing']:
        print('   Route -> MEMORY SAVER (Draft passed!)')
        return 'save_memory_node'
    elif state['iterations'] >= 3:
        print('   Route -> END (Max iterations reached)')
        return END
    else:
        print('   Route -> ACTOR (Draft failed, try again)')
        return 'actor_node'

def save_memory_node(state: ReflexionState):
    print('\n[MEMORY] Saving successful draft to episodic_memory.json ...')
    save_memory(task=state['task'], successful_result=state['current_draft'], critique=state['critique'])
    return state

print('Graph nodes defined.')

### Build & Run the Reflexion Graph

In [ ]:
graph = StateGraph(ReflexionState)
graph.add_node('actor_node', actor_node)
graph.add_node('evaluator_node', evaluator_node)
graph.add_node('save_memory_node', save_memory_node)
graph.add_edge(START, 'actor_node')
graph.add_edge('actor_node', 'evaluator_node')
graph.add_conditional_edges('evaluator_node', route_evaluation)
graph.add_edge('save_memory_node', END)

app = graph.compile()

task = (
    'Draft an email to a highly demanding client apologizing for a 2-week delay '
    'on the Q3 report. It must be extremely professional, under 100 words, '
    'and have a clear greeting, body, and sign-off.'
)

initial_state = {
    'task': task,
    'current_draft': '',
    'critique': '',
    'is_passing': False,
    'iterations': 0,
}

print('Starting the Reflexion loop ...')
print(f'   Task: {task}\n')

app.invoke(initial_state)

print('\nKEY TAKEAWAYS:')
print('   1. ACTOR-EVALUATOR PARADIGM — The agent critiques its own work.')
print('   2. RETRY LOOP — It uses the critique to rewrite.')
print('   3. EPISODIC MEMORY — It saved the final passing draft to disk.')
print('\n   Run this again! It will likely PASS on the first try because it learned!')